### Step 1: Import Libraries

In [23]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder

### Step 2: Load Dataset

In [24]:
df = pd.read_csv('adult_with_headers.csv')
# Display first 5 rows
print("First 5 rows:\n")
df.head()

First 5 rows:



,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [26]:
df.shape

(32561, 15)

### Step 3: Basic Data Exploration

In [29]:
print("\nDataset Info:")
df.info()

print("\nSummary Statistics:")
df.describe()

print("\nMissing Values Before Cleaning:")
df.isnull().sum()



Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32561 entries, 0 to 32560
Data columns (total 21 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   age                       32561 non-null  int64
 1   workclass                 32561 non-null  int64
 2   fnlwgt                    32561 non-null  int64
 3   education                 32561 non-null  int64
 4   education_num             32561 non-null  int64
 5   marital_status            32561 non-null  int64
 6   occupation                32561 non-null  int64
 7   relationship              32561 non-null  int64
 8   capital_gain              32561 non-null  int64
 9   capital_loss              32561 non-null  int64
 10  hours_per_week            32561 non-null  int64
 11  native_country            32561 non-null  int64
 12  race_ Amer-Indian-Eskimo  32561 non-null  bool 
 13  race_ Asian-Pac-Islander  32561 non-null  bool 
 14  race_ Black            

age                         0
workclass                   0
fnlwgt                      0
education                   0
education_num               0
marital_status              0
occupation                  0
relationship                0
capital_gain                0
capital_loss                0
hours_per_week              0
native_country              0
race_ Amer-Indian-Eskimo    0
race_ Asian-Pac-Islander    0
race_ Black                 0
race_ Other                 0
race_ White                 0
sex_ Female                 0
sex_ Male                   0
income_ <=50K               0
income_ >50K                0
dtype: int64

### Step 4: Handle Missing Values

In [7]:
df.replace(' ?', np.nan, inplace=True)

In [8]:
print("\nMissing Values After Replacing '?':")
print(df.isnull().sum())


Missing Values After Replacing '?':
age                  0
workclass         1836
fnlwgt               0
education            0
education_num        0
marital_status       0
occupation        1843
relationship         0
race                 0
sex                  0
capital_gain         0
capital_loss         0
hours_per_week       0
native_country     583
income               0
dtype: int64


In [9]:
# Separate numerical and categorical columns
num_cols = df.select_dtypes(include=['int64', 'float64']).columns
cat_cols = df.select_dtypes(include=['object']).columns

In [10]:
# Fill numerical columns with median
for col in num_cols:
    df[col].fillna(df[col].median(), inplace=True)

/var/folders/c6/rh118pnd4z74yrqrmh0p58mc0000gn/T/ipykernel_51160/1373254087.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].median(), inplace=True)


In [14]:
for col in cat_cols:
    df[col].fillna(df[col].mode()[0], inplace=True)

print("\nMissing Values After Imputation:")
print(df.isnull().sum())


Missing Values After Imputation:
age               0
workclass         0
fnlwgt            0
education         0
education_num     0
marital_status    0
occupation        0
relationship      0
race              0
sex               0
capital_gain      0
capital_loss      0
hours_per_week    0
native_country    0
income            0
dtype: int64


/var/folders/c6/rh118pnd4z74yrqrmh0p58mc0000gn/T/ipykernel_51160/378568879.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].mode()[0], inplace=True)


### Step 5: Feature Engineering

In [16]:
# Create Age Group Feature
df['age_group'] = pd.cut(df['age'], bins=[0, 25, 50, 100], 
                         labels=['Young', 'Adult', 'Senior'])

# Create Work Intensity Feature
df['work_intensity'] = pd.cut(df['hours_per_week'], 
                              bins=[0, 25, 40, 100], 
                              labels=['Part-time', 'Full-time', 'Overtime'])

print("\nNew Features Added:")
print(df[['age_group', 'work_intensity']].head())



New Features Added:
  age_group work_intensity
0     Adult      Full-time
1     Adult      Part-time
2     Adult      Full-time
3    Senior      Full-time
4     Adult      Full-time


### Step 6: Handle Skewness (Log Transformation)

In [19]:
print("\nSkewness Before Transformation:")
print(df[num_cols].skew())

# Apply log transformation on 'capital-gain' (skewed feature)
df['capital_gain'] = np.log1p(df['capital_gain'])

print("\nSkewness After Transformation:")
print(df['capital_gain'].skew())



Skewness Before Transformation:
age                0.558743
fnlwgt             1.446980
education_num     -0.311676
capital_gain      11.953848
capital_loss       4.594629
hours_per_week     0.227643
dtype: float64

Skewness After Transformation:
3.096143524467517


### Step 7: Encoding

In [28]:
# Update categorical columns after feature engineering
cat_cols = df.select_dtypes(include=['object', 'category']).columns

# Separate low and high cardinality columns
low_card_cols = [col for col in cat_cols if df[col].nunique() <= 5]
high_card_cols = [col for col in cat_cols if df[col].nunique() > 5]

# One-Hot Encoding for low cardinality
df = pd.get_dummies(df, columns=low_card_cols)

# Label Encoding for high cardinality
le = LabelEncoder()
for col in high_card_cols:
    df[col] = le.fit_transform(df[col])

print("\nData After Encoding:")
df.head()


Data After Encoding:


,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,capital_gain,capital_loss,...,native_country,race_ Amer-Indian-Eskimo,race_ Asian-Pac-Islander,race_ Black,race_ Other,race_ White,sex_ Female,sex_ Male,income_ <=50K,income_ >50K
0,39,7,77516,9,13,4,1,1,2174,0,...,39,False,False,False,False,True,False,True,True,False
1,50,6,83311,9,13,2,4,0,0,0,...,39,False,False,False,False,True,False,True,True,False
2,38,4,215646,11,9,0,6,1,0,0,...,39,False,False,False,False,True,False,True,True,False
3,53,4,234721,1,7,2,6,0,0,0,...,39,False,False,True,False,False,False,True,True,False
4,28,4,338409,9,13,2,10,5,0,0,...,5,False,False,True,False,False,True,False,True,False


### Step 8: Scaling

In [32]:
# Apply Standard Scaling
scaler_standard = StandardScaler()
df_standard_scaled = df.copy()
df_standard_scaled[num_cols] = scaler_standard.fit_transform(df_standard_scaled[num_cols])

print("\nStandard Scaled Data Sample:")
df_standard_scaled.head()


Standard Scaled Data Sample:


,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,capital_gain,capital_loss,...,native_country,race_ Amer-Indian-Eskimo,race_ Asian-Pac-Islander,race_ Black,race_ Other,race_ White,sex_ Female,sex_ Male,income_ <=50K,income_ >50K
0,0.030671,7,-1.063611,9,1.134739,4,1,1,0.148453,-0.21666,...,39,False,False,False,False,True,False,True,True,False
1,0.837109,6,-1.008707,9,1.134739,2,4,0,-0.145920,-0.21666,...,39,False,False,False,False,True,False,True,True,False
2,-0.042642,4,0.245079,11,-0.420060,0,6,1,-0.145920,-0.21666,...,39,False,False,False,False,True,False,True,True,False
3,1.057047,4,0.425801,1,-1.197459,2,6,0,-0.145920,-0.21666,...,39,False,False,True,False,False,False,True,True,False
4,-0.775768,4,1.408176,9,1.134739,2,10,5,-0.145920,-0.21666,...,5,False,False,True,False,False,True,False,True,False


In [33]:
# Apply Min-Max Scaling
scaler_minmax = MinMaxScaler()
df_minmax_scaled = df.copy()
df_minmax_scaled[num_cols] = scaler_minmax.fit_transform(df_minmax_scaled[num_cols])

print("\nMin-Max Scaled Data Sample:")
df_minmax_scaled.head()



Min-Max Scaled Data Sample:


,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,capital_gain,capital_loss,...,native_country,race_ Amer-Indian-Eskimo,race_ Asian-Pac-Islander,race_ Black,race_ Other,race_ White,sex_ Female,sex_ Male,income_ <=50K,income_ >50K
0,0.301370,7,0.044302,9,0.800000,4,1,1,0.02174,0.0,...,39,False,False,False,False,True,False,True,True,False
1,0.452055,6,0.048238,9,0.800000,2,4,0,0.00000,0.0,...,39,False,False,False,False,True,False,True,True,False
2,0.287671,4,0.138113,11,0.533333,0,6,1,0.00000,0.0,...,39,False,False,False,False,True,False,True,True,False
3,0.493151,4,0.151068,1,0.400000,2,6,0,0.00000,0.0,...,39,False,False,True,False,False,False,True,True,False
4,0.150685,4,0.221488,9,0.800000,2,10,5,0.00000,0.0,...,5,False,False,True,False,False,True,False,True,False


### Step 9 : Final Output

In [22]:
print("\nFinal Dataset Shape:", df.shape)

# Save processed dataset
df.to_csv('processed_adult.csv', index=False)

print("\nPreprocessing Completed Successfully!")


Final Dataset Shape: (32561, 27)

Preprocessing Completed Successfully!
